# Homework 2

# Set up

## Installing packages

In [ ]:
!pip install requests PyPDF2 gdown
!pip install 'markitdown[pdf]'
!pip install langchain_mcp_adapters langchain_google_genai langchain-openai

## Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `VERTEX_API_KEY`.


1.   Look for the key icon on the left panel of your colab.
2.   Under `Name`, create `VERTEX_API_KEY`.
3. Copy your key to `Value`.

If you cannot use VERTEX_API_KEY, you can use deepseek models via `DEEPSEEK_API_KEY`. It does not affect your score.



In [ ]:
from google.colab import userdata
GEMINI_VERTEX_API_KEY = userdata.get('VERTEX_API_KEY')
# DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')

# Download sample CVs

## Downloading sample_cv.pdf
The codes below download the sample CV


In [ ]:
import os
import gdown

folder_id = "1adYKq7gSSczFP3iikfA8Er-HSZP6VM7D"
folder_url = f"https://drive.google.com/drive/folders/{folder_id}"

output_dir = "downloaded_cvs"
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=folder_url,
    output=output_dir,
    quiet=False,
    use_cookies=False
)

In [ ]:
# =====================================================
#  Load and display all CV PDFs in order
# =====================================================
import os
from markitdown import MarkItDown

cv_dir = "downloaded_cvs"

# Initialize MarkItDown
md = MarkItDown(enable_plugins=False)

# Collect and sort PDFs numerically
pdf_files = sorted(
    [f for f in os.listdir(cv_dir) if f.lower().endswith(".pdf")],
    key=lambda x: int("".join(filter(str.isdigit, x)))  # CV_1.pdf → 1
)

all_cvs = []

for pdf_name in pdf_files:
    pdf_path = os.path.join(cv_dir, pdf_name)
    result = md.convert(pdf_path)

    all_cvs.append({
        "file": pdf_name,
        "text": result.text_content
    })

    print("=" * 80)
    print(f"📄 {pdf_name}")
    print("=" * 80)
    print(result.text_content)
    print("\n\n")


# Connect to our MCP server

Documentation about MCP: https://modelcontextprotocol.io/docs/getting-started/intro.

Using MCP servers in Langchain https://docs.langchain.com/oss/python/langchain/mcp.

## Check which tools that the MCP server provide

In [ ]:
import asyncio
import json
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})

mcp_tools = await client.get_tools()
for tool in mcp_tools:
    print(tool.name)
    print(tool.description)
    print(tool.args)
    print("\n\n------------------------------------------------------\n\n")

tools = mcp_tools

## A simple agent using tools from the MCP server


In [ ]:
import os
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mcp_adapters.client import MultiServerMCPClient

# ---------------------------
# 1. Define a local tool
# ---------------------------
@tool
def say_hello(name: str) -> str:
    """Say hello to a person by name."""
    return f"Hello, {name}! 👋"

# ---------------------------
# 2. Load MCP tools + merge
# ---------------------------
client = MultiServerMCPClient({
    "social_graph": {
        "transport": "http",
        "url": "https://ftec5660.ngrok.app/mcp",
        "headers": {"ngrok-skip-browser-warning": "true"}
    }
})

mcp_tools = await client.get_tools()
tools = mcp_tools + [say_hello]

# ---------------------------
# 3. Initialize Gemini (tool-enabled) or deepseek
# ---------------------------
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=GEMINI_VERTEX_API_KEY,
    vertexai=True,
    temperature=0,
)

# from langchain_openai import ChatOpenAI
# DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
# llm = ChatOpenAI(
#     model="deepseek-chat",          # or "deepseek-reasoner"
#     api_key=DEEPSEEK_API_KEY,
#     base_url="https://api.deepseek.com/v1",
#     temperature=0,
# )

llm_with_tools = llm.bind_tools(tools)

# ---------------------------
# 4. Single-step invocation
# ---------------------------
query = "Say hello to Bao using tool, then search for someone named Alice on Facebook."

response = llm_with_tools.invoke([
    HumanMessage(content=query)
])

print(response)

# CV Extraction

In [ ]:
import json

def extract_cv_info(cv_text: str) -> dict:
    """Extract structured information from CV text using LLM"""

    prompt = f"""You are a CV information extraction assistant. Extract the following information from the CV below:

1. name: Full name of the candidate
2. current_location: Current city and country
3. education: List of education entries (school, degree, field, graduation_year)
4. experience: List of work experiences (company, title, start_year, end_year)
5. skills: List of skills mentioned

Return a JSON object with these fields. If information is not available, use null or empty list.

CV Text:
{cv_text}

Return ONLY valid JSON, no additional text."""

    response = llm.invoke(prompt)

    try:
        content = response.content
        # Extract JSON from response
        if "```json" in content:
            content = content.split("```json")[1].split("```")[0]
        elif "```" in content:
            content = content.split("```")[1].split("```")[0]

        return json.loads(content.strip())
    except Exception as e:
        print(f"Error extracting CV info: {e}")
        return {
            "name": None,
            "current_location": None,
            "education": [],
            "experience": [],
            "skills": []
        }

# Test extraction on first CV
test_cv_info = extract_cv_info(all_cvs[0]["text"])
print("Test extraction result:")
print(json.dumps(test_cv_info, indent=2))


In [ ]:
# Assign MCP tools to variables
search_facebook_users = mcp_tools[0]
get_facebook_profile = mcp_tools[1]
get_facebook_mutual_friends = mcp_tools[2]
search_linkedin_people = mcp_tools[3]
get_linkedin_profile = mcp_tools[4]
get_linkedin_interactions = mcp_tools[5]

# Profile search&match

In [ ]:
import json
import asyncio
from langchain_core.messages import HumanMessage

# Helper function to parse MCP tool results
def parse_mcp_result(result, default="[]"):
    """
    Parse MCP tool result which can be in different formats:
    - Direct list/dict
    - List containing a dict with 'text' field
    - List containing a string
    """
    if result is None:
        return default

    # If it's already a list
    if isinstance(result, list):
        if len(result) == 0:
            return default
        # Check if first element has 'text' field
        if isinstance(result[0], dict) and 'text' in result[0]:
            return result[0].get('text', default)
        # If first element is a string
        if isinstance(result[0], str):
            return result[0]
        # Otherwise, convert the whole list to JSON string
        return json.dumps(result)

    # If it's a dict
    if isinstance(result, dict):
        if 'text' in result:
            return result.get('text', default)
        return json.dumps(result)

    # If it's a string
    if isinstance(result, str):
        return result

    return default

# Create LLM with tools
llm_with_tools = llm.bind_tools(mcp_tools)

async def search_and_match_profiles(cv_info: dict) -> dict:
    """
    Search for social media profiles that match the CV information.
    Returns the best matched Facebook and LinkedIn profiles.
    """
    def parse_llm_json_response(content):
        """安全解析 LLM 返回的 JSON"""
        if content is None:
            return None
        if isinstance(content, list):
            if len(content) > 0:
                return parse_llm_json_response(content[0])
            return None
        if isinstance(content, str):
            try:
                return json.loads(content)
            except:
                for marker in ["```json", "```"]:
                    if marker in content:
                        try:
                            return json.loads(content.split(marker)[1].split("```")[0])
                        except:
                            pass
                return None
        return None

    name = cv_info.get("name", "")
    if not name:
        return {"facebook": None, "linkedin": None, "discrepancies": []}

    print(f"\n🔍 Searching for: {name}")

    # Try to get location for better search
    location = cv_info.get("current_location", "")

    results = {
        "facebook": None,
        "linkedin": None,
        "discrepancies": [],
        "match_details": {}
    }

    # ===== Facebook Search =====
    try:
        # Search Facebook users
        fb_search_result = await search_facebook_users.ainvoke({
            "q": name,
            "limit": 5,
            "fuzzy": True
        })

        # Parse MCP result properly
        fb_text = parse_mcp_result(fb_search_result, '[]')
        fb_candidates = json.loads(fb_text)

        if fb_candidates and len(fb_candidates) > 0:
            print(f"   📘 Found {len(fb_candidates)} Facebook candidates")

            # Get detailed profiles for top candidates
            fb_profiles = []
            for candidate in fb_candidates[:3]:
                try:
                    profile_result = await get_facebook_profile.ainvoke({
                        "user_id": candidate["id"]
                    })
                    # Parse profile result properly
                    profile_text = parse_mcp_result(profile_result, '{}')
                    profile = json.loads(profile_text)
                    if "error" not in profile:
                        fb_profiles.append(profile)
                except Exception as e:
                    print(f"   ⚠️ Error getting FB profile: {e}")
                    continue

            # Use LLM to find best match
            if fb_profiles:
                match_prompt = f"""Given CV info: {json.dumps(cv_info)}

Facebook candidates: {json.dumps(fb_profiles, ensure_ascii=False)}

Which candidate matches BEST with the CV? Consider:
- Name similarity
- Location match
- Job title similarity
- Education similarity

CRITICAL: You MUST respond with ONLY valid JSON, no other text. Format:
{{"best_index": 0, "reason": "brief explanation", "confidence": 0.9}}

If no candidate matches, use best_index: -1"""

                response = llm.invoke([HumanMessage(content=match_prompt)])

                try:
                    result = parse_llm_json_response(response.content)

                    if result is None:
                        print(f"    Failed to parse FB match response - LLM returned non-JSON response")
                        print(f"    Raw response type: {type(response.content)}")
                        if isinstance(response.content, list):
                            print(f"    Raw response content: {response.content}")
                        elif isinstance(response.content, str):
                            print(f"    Raw response (first 500 chars): {response.content[:500]}")
                    elif not isinstance(result, dict):
                        print(f"   Failed to parse FB match response - got {type(result)}")
                    else:
                        idx = result.get('best_index', -1)
                        confidence = result.get('confidence', 0)

                        if 0 <= idx < len(fb_profiles):
                            results["facebook"] = fb_profiles[idx]
                            results["match_details"]["facebook_confidence"] = confidence
                            print(f"   📘 Best FB match: {fb_profiles[idx].get('display_name')} (confidence: {confidence})")
                except Exception as e:
                    print(f"    Error parsing FB match: {e}")
    except Exception as e:
        print(f"   Facebook search error: {e}")

    # ===== LinkedIn Search =====
    try:
        # Search LinkedIn users
        li_search_params = {"q": name, "limit": 5, "fuzzy": True}
        if location:
            loc_parts = location.split(",")
            if len(loc_parts) > 0 and loc_parts[0].strip():
                li_search_params["location"] = loc_parts[0].strip()

        li_search_result = await search_linkedin_people.ainvoke(li_search_params)

        # Parse MCP result properly
        li_text = parse_mcp_result(li_search_result, '[]')
        li_candidates = json.loads(li_text)

        if li_candidates and len(li_candidates) > 0:
            print(f"   💼 Found {len(li_candidates)} LinkedIn candidates")

            # Get detailed profiles
            li_profiles = []
            for candidate in li_candidates[:3]:
                try:
                    profile_result = await get_linkedin_profile.ainvoke({
                        "person_id": candidate["id"]
                    })
                    # Parse profile result properly
                    profile_text = parse_mcp_result(profile_result, '{}')
                    profile = json.loads(profile_text)
                    if "error" not in profile:
                        li_profiles.append(profile)
                except Exception as e:
                    print(f"    Error getting LI profile: {e}")
                    continue

            # Use LLM to find best match
            if li_profiles:
                match_prompt = f"""Given CV info: {json.dumps(cv_info)}

LinkedIn candidates: {json.dumps(li_profiles, ensure_ascii=False)}

Which candidate matches BEST with the CV? Consider:
- Name similarity
- Location match
- Job title similarity
- Work experience match
- Education match

CRITICAL: You MUST respond with ONLY valid JSON, no other text. Format:
{{"best_index": 0, "reason": "brief explanation", "confidence": 0.9}}

If no candidate matches, use best_index: -1"""

                response = llm.invoke([HumanMessage(content=match_prompt)])

                try:
                    result = parse_llm_json_response(response.content)
                    if result is None:
                        print(f"    Failed to parse LI match response - LLM returned non-JSON response")
                        print(f"    Raw response type: {type(response.content)}")
                        if isinstance(response.content, list):
                            print(f"    Raw response content: {response.content}")
                        elif isinstance(response.content, str):
                            print(f"    Raw response (first 500 chars): {response.content[:500]}")
                    elif not isinstance(result, dict):
                        print(f"    Failed to parse LI match response - got {type(result)}")
                    else:
                        idx = result.get('best_index', -1)
                        confidence = result.get('confidence', 0)

                        if 0 <= idx < len(li_profiles):
                            results["linkedin"] = li_profiles[idx]
                            results["match_details"]["linkedin_confidence"] = confidence
                            print(f"   💼 Best LI match: {li_profiles[idx].get('name')} (confidence: {confidence})")
                except Exception as e:
                    print(f"    Error parsing LI match: {e}")
    except Exception as e:
        print(f"    LinkedIn search error: {e}")

    return results

print("✅ Profile search and matching function defined")


# Discrepancy detection and scoring function defined

In [ ]:
async def detect_discrepancies(cv_info: dict, matched_profiles: dict) -> dict:
    """
    Compare CV information with matched social media profiles to detect discrepancies.
    Returns detailed discrepancy report and a score.
    """
    discrepancies = []
    match_details = matched_profiles.get("match_details", {})

    facebook = matched_profiles.get("facebook")
    linkedin = matched_profiles.get("linkedin")

    # ===== Check Facebook discrepancies =====
    if facebook:
        fb_name = facebook.get("display_name", "")
        cv_name = cv_info.get("name", "")

        # Check name discrepancy
        if cv_name and fb_name:
            name_prompt = f"""Compare these two names:
1. "{cv_name}"
2. "{fb_name}"

Are they the same person? Consider:
- Same first and last name
- Nicknames or variations
- Different name might indicate different person

Respond with JSON:
{{"is_same": true/false, "reason": "brief explanation"}}"""

            try:
                response = llm.invoke([HumanMessage(content=name_prompt)])
                result = parse_llm_json_response(response.content)
                if result is not None and not result.get("is_same", True):
                    discrepancies.append({
                        "type": "name_mismatch",
                        "source": "facebook",
                        "cv_value": cv_name,
                        "profile_value": fb_name,
                        "severity": "high",
                        "description": f"Name mismatch: CV shows '{cv_name}' but FB shows '{fb_name}'"
                    })
            except:
                pass

        # Check location discrepancy
        cv_location = cv_info.get("current_location", "")
        fb_city = facebook.get("city", "")
        fb_country = facebook.get("country", "")

        if cv_location and (fb_city or fb_country):
            location_prompt = f"""Compare these locations:
CV location: "{cv_location}"
Facebook profile: city="{fb_city}", country="{fb_country}"

Are they in the same region? Consider:
- Same city
- Same country
- Very different locations might indicate different person

Respond with JSON:
{{"is_match": true/false, "reason": "brief explanation"}}"""

            try:
                response = llm.invoke([HumanMessage(content=location_prompt)])
                result = parse_llm_json_response(response.content)
                if result is not None and not result.get("is_match", True):
                    discrepancies.append({
                        "type": "location_mismatch",
                        "source": "facebook",
                        "cv_value": cv_location,
                        "profile_value": f"{fb_city}, {fb_country}",
                        "severity": "medium",
                        "description": f"Location mismatch: CV shows '{cv_location}' but FB shows '{fb_city}, {fb_country}'"
                    })
            except:
                pass

    # ===== Check LinkedIn discrepancies =====
    if linkedin:
        li_name = linkedin.get("name", "")
        cv_name = cv_info.get("name", "")

        # Check name discrepancy
        if cv_name and li_name:
            name_prompt = f"""Compare these two names:
1. "{cv_name}"
2. "{li_name}"

Are they the same person? Consider:
- Same first and last name
- Nicknames or variations

Respond with JSON:
{{"is_same": true/false, "reason": "brief explanation"}}"""

            try:
                response = llm.invoke([HumanMessage(content=name_prompt)])
                result = parse_llm_json_response(response.content)
                if result is not None and not result.get("is_same", True):
                    discrepancies.append({
                        "type": "name_mismatch",
                        "source": "linkedin",
                        "cv_value": cv_name,
                        "profile_value": li_name,
                        "severity": "high",
                        "description": f"Name mismatch: CV shows '{cv_name}' but LinkedIn shows '{li_name}'"
                    })
            except:
                pass

        # Check work experience discrepancy
        cv_experience = cv_info.get("experience", [])
        li_experience = linkedin.get("experience", [])

        if cv_experience and li_experience:
            exp_prompt = f"""Compare work experiences:

CV Experience:
{json.dumps(cv_experience, ensure_ascii=False)}

LinkedIn Experience:
{json.dumps(li_experience, ensure_ascii=False)}

Find significant discrepancies:
- Different companies
- Different job titles
- Different time periods
- Major position differences

Respond with JSON with list of discrepancies:
{{"discrepancies": [{{"type": "...", "cv_value": "...", "li_value": "...", "severity": "high/medium/low"}}]}}"""

            try:
                response = llm.invoke([HumanMessage(content=exp_prompt)])
                result = parse_llm_json_response(response.content)
                exp_discrepancies = result.get("discrepancies", []) if result else []
                for d in exp_discrepancies:
                    d["source"] = "linkedin"
                    d["description"] = f"Experience discrepancy: {d.get('cv_value')} vs {d.get('li_value')}"
                discrepancies.extend(exp_discrepancies)
            except:
                pass

        # Check education discrepancy
        cv_education = cv_info.get("education", [])
        li_education = linkedin.get("education", [])

        if cv_education and li_education:
            edu_prompt = f"""Compare education:

CV Education:
{json.dumps(cv_education, ensure_ascii=False)}

LinkedIn Education:
{json.dumps(li_education, ensure_ascii=False)}

Find discrepancies:
- Different schools/universities
- Different degrees
- Different fields of study
- Different graduation years (more than 2 years is significant)

Respond with JSON with list of discrepancies:
{{"discrepancies": [{{"type": "...", "cv_value": "...", "li_value": "...", "severity": "high/medium/low"}}]}}"""

            try:
                response = llm.invoke([HumanMessage(content=edu_prompt)])
                result = parse_llm_json_response(response.content)
                edu_discrepancies = result.get("discrepancies", []) if result else []
                for d in edu_discrepancies:
                    d["source"] = "linkedin"
                    d["description"] = f"Education discrepancy: {d.get('cv_value')} vs {d.get('li_value')}"
                discrepancies.extend(edu_discrepancies)
            except:
                pass

    # ===== Calculate Score =====
    # Base score starts at 0.5
    score = 0.35

    # Adjust based on profile matches
    # LinkedIn is more reliable than Facebook for professional verification
    fb_confidence = match_details.get("facebook_confidence", 0)
    li_confidence = match_details.get("linkedin_confidence", 0)

    if facebook and linkedin:
        # Both platforms matched - most reliable
        score += fb_confidence * 0.2 + li_confidence * 0.3
    elif linkedin:
        # Only LinkedIn - still good
        score += li_confidence * 0.3
    elif facebook:
        # Only Facebook - less reliable, apply heavy penalty
        score += fb_confidence * 0.1
    # No match: score stays at 0.35
    # # 如果只有FB匹配但没有LI匹配，应该扣分
    # if facebook and not linkedin:
    #     discrepancies.append({
    #         "type": "missing_linkedin",
    #         "severity": "high",
    #         "description": "Found Facebook match but no LinkedIn profile - LinkedIn is more reliable for professional verification"
    #     })
    # Adjust for discrepancies
    high_severity = sum(1 for d in discrepancies if d.get("severity") == "high")
    medium_severity = sum(1 for d in discrepancies if d.get("severity") == "medium")
    low_severity = sum(1 for d in discrepancies if d.get("severity") == "low")

    score -= high_severity * 0.2
    score -= medium_severity * 0.1
    score -= low_severity * 0.05

    # Ensure score is in [0, 1]
    score = max(0.0, min(1.0, score))

    return {
        "discrepancies": discrepancies,
        "score": score,
        "has_facebook_match": facebook is not None,
        "has_linkedin_match": linkedin is not None,
        "high_severity_count": high_severity,
        "medium_severity_count": medium_severity,
        "low_severity_count": low_severity
    }

print("✅ Discrepancy detection and scoring function defined")


# Generate verification report

In [ ]:
# =====================================================
#  Verification Report Generator
# =====================================================
def generate_verification_report(cv_info: dict, matched_profiles: dict, verification_result: dict) -> dict:
    """
    Generate a comprehensive verification report for a CV.
    """

    report = {
        "summary": {
            "candidate_name": cv_info.get("name", "Unknown"),
            "claimed_location": cv_info.get("current_location", "Unknown"),
            "education_count": len(cv_info.get("education", [])),
            "experience_count": len(cv_info.get("experience", [])),
            "skills_count": len(cv_info.get("skills", []))
        },
        "social_media_matches": {
            "facebook": {"found": verification_result.get("has_facebook_match", False), "profile": None},
            "linkedin": {"found": verification_result.get("has_linkedin_match", False), "profile": None}
        },
        "discrepancies": {
            "total": len(verification_result.get("discrepancies", [])),
            "high_severity": verification_result.get("high_severity_count", 0),
            "medium_severity": verification_result.get("medium_severity_count", 0),
            "low_severity": verification_result.get("low_severity_count", 0),
            "details": verification_result.get("discrepancies", [])
        },
        "score": {
            "value": verification_result.get("score", 0.0),
            "decision": "Valid CV" if verification_result.get("score", 0) > 0.5 else "Problematic CV"
        }
    }

    # Add matched profile details
    if matched_profiles.get("facebook"):
        fb = matched_profiles["facebook"]
        report["social_media_matches"]["facebook"]["profile"] = {
            "display_name": fb.get("display_name"),
            "location": f"{fb.get('city', '')}, {fb.get('country', '')}",
            "original_name": fb.get("original_name"),
            "job_title": fb.get("job_title"),
            "company": fb.get("company")
        }

    if matched_profiles.get("linkedin"):
        li = matched_profiles["linkedin"]
        report["social_media_matches"]["linkedin"]["profile"] = {
            "name": li.get("name"),
            "headline": li.get("headline"),
            "location": li.get("location"),
            "experience": li.get("experience", [])[:2],
            "education": li.get("education", [])[:2]
        }

    return report


def print_verification_report(report: dict, cv_file: str):
    """Print a formatted verification report"""

    print(f"\n{'='*80}")
    print(f"📋 VERIFICATION REPORT: {cv_file}")
    print(f"{'='*80}")

    # Summary
    print("\n📌 SUMMARY")
    print("-" * 40)
    summary = report["summary"]
    print(f"  Candidate Name: {summary['candidate_name']}")
    print(f"  Claimed Location: {summary['claimed_location']}")
    print(f"  Education: {summary['education_count']} entries")
    print(f"  Experience: {summary['experience_count']} entries")

    # Social Media Matches
    print("\n📘 SOCIAL MEDIA MATCHES")
    print("-" * 40)

    fb = report["social_media_matches"]["facebook"]
    print(f"  Facebook: {'✓ Found' if fb['found'] else '✗ Not Found'}")
    if fb["profile"]:
        p = fb["profile"]
        print(f"    - Name: {p.get('display_name', 'N/A')}")
        print(f"    - Location: {p.get('location', 'N/A')}")

    li = report["social_media_matches"]["linkedin"]
    print(f"  LinkedIn: {'✓ Found' if li['found'] else '✗ Not Found'}")
    if li["profile"]:
        p = li["profile"]
        print(f"    - Name: {p.get('name', 'N/A')}")
        print(f"    - Headline: {p.get('headline', 'N/A')}")

    # Discrepancies
    print("\n⚠️ DISCREPANCIES")
    print("-" * 40)
    disc = report["discrepancies"]
    print(f"  Total: {disc['total']} | High: {disc['high_severity']} | Medium: {disc['medium_severity']} | Low: {disc['low_severity']}")

    if disc["details"]:
        for i, d in enumerate(disc["details"], 1):
            print(f"    {i}. [{d.get('severity', '?').upper()}] {d.get('description', '')[:60]}...")
    else:
        print("  ✅ No discrepancies found")

    # Score
    print("\n📊 SCORE & DECISION")
    print("-" * 40)
    score = report["score"]
    print(f"  Score: {score['value']:.2f} / 1.00")
    print(f"  Decision: {score['decision']}")
    print(f"\n{'='*80}\n")


def generate_all_reports(all_results: list):
    """Generate and print reports for all CVs"""
    all_reports = []

    for result in all_results:
        report = generate_verification_report(
            cv_info=result["cv_info"],
            matched_profiles=result["matched_profiles"],
            verification_result=result["verification_result"]
        )
        report["file"] = result["file"]
        all_reports.append(report)

        # Print the report
        print_verification_report(report, result["file"])

    return all_reports


print("✅ Verification report generator defined")


# Process

In [ ]:
# Process all CVs and generate scores
async def process_all_cvs(all_cvs: list) -> list:
    """
    Process all CVs and return verification scores.
    """
    scores = []
    detailed_results = []

    for cv in all_cvs:
        print(f"\n{'='*80}")
        print(f"📄 Processing: {cv['file']}")
        print(f"{'='*80}")

        # Step 1: Extract CV information
        print("\n📝 Step 1: Extracting CV information...")
        cv_info = extract_cv_info(cv['text'])
        print(f"   Name: {cv_info.get('name', 'N/A')}")
        print(f"   Location: {cv_info.get('current_location', 'N/A')}")
        edu_count = len(cv_info.get('education', []))
        exp_count = len(cv_info.get('experience', []))
        print(f"   Education: {edu_count} entry{'s' if edu_count != 1 else ''}")
        print(f"   Experience: {exp_count} entry{'s' if exp_count != 1 else ''}")

        # Step 2: Search and match social media profiles
        print("\n🔍 Step 2: Searching and matching social media profiles...")
        matched_profiles = await search_and_match_profiles(cv_info)

        # Step 3: Detect discrepancies and calculate score
        print("\n⚖️ Step 3: Detecting discrepancies and calculating score...")
        verification_result = await detect_discrepancies(cv_info, matched_profiles)

        score = verification_result["score"]
        discrepancies = verification_result["discrepancies"]

        print(f"\n   📊 Score: {score:.2f}")
        print(f"   📘 Facebook match: {'Yes' if verification_result['has_facebook_match'] else 'No'}")
        print(f"   💼 LinkedIn match: {'Yes' if verification_result['has_linkedin_match'] else 'No'}")

        if discrepancies:
            print(f"\n   ⚠️ Discrepancies found: {len(discrepancies)}")
            for d in discrepancies[:5]:  # Show first 5
                print(f"      - [{d.get('severity', '?')}] {d.get('description', 'N/A')}")
        else:
            print(f"\n   ✅ No discrepancies found")

        scores.append(score)
        detailed_results.append({
            "file": cv['file'],
            "cv_info": cv_info,
            "matched_profiles": matched_profiles,
            "verification_result": verification_result
        })

        # Small delay to avoid rate limiting
        await asyncio.sleep(1)

    return scores, detailed_results

# Run the verification process
print("🚀 Starting CV verification process...")
print("="*80)

all_scores, all_results = await process_all_cvs(all_cvs)

# =====================================================
#  Generate and Display Verification Reports
# =====================================================
print("\n" + "="*80)
print("📋 GENERATING DETAILED VERIFICATION REPORTS")
print("="*80)

all_reports = generate_all_reports(all_results)

# Also output JSON format
print("\n" + "="*80)
print("📝 REPORTS IN JSON FORMAT")
print("="*80)

import json
for report in all_reports:
    print(f"\n--- {report['file']} ---")
    print(json.dumps({
        "file": report["file"],
        "candidate_name": report["summary"]["candidate_name"],
        "facebook_found": report["social_media_matches"]["facebook"]["found"],
        "linkedin_found": report["social_media_matches"]["linkedin"]["found"],
        "discrepancies": report["discrepancies"]["total"],
        "score": report["score"]["value"],
        "decision": report["score"]["decision"]
    }, indent=2))

print("\n" + "="*80)
print("📊 FINAL RESULTS")
print("="*80)

for i, (score, result) in enumerate(zip(all_scores, all_results)):
    print(f"\nCV_{i+1}.pdf: Score = {score:.2f}")
    print(f"   - Decision: {'Valid CV' if score > 0.5 else 'Problematic CV'}")

# Format output as required
print("\n" + "="*80)
print("📝 Output Format:")
print("="*80)
print(f"scores = {all_scores}")
print("\n# Ground truth for evaluation (DO NOT MODIFY):")
print("groundtruth = [1, 1, 1, 0, 0]")


# Evaluation code

In the test phase, you will be given 5 CV files with fixed names:

    CV_1.pdf, CV_2.pdf, CV_3.pdf, CV_4.pdf, CV_5.pdf

Your system must process these CVs and output a list of 5 scores,
one score per CV, in the same order:

    scores = [s1, s2, s3, s4, s5]

Each score must be a float in the range [0, 1], representing the
reliability or confidence that the CV is valid (or meets the task criteria).

The ground-truth labels are binary:

    groundtruth = [0 or 1, ..., 0 or 1]

Each CV is evaluated independently using a threshold of 0.5:

- If score > 0.5 and groundtruth == 1 → Full credit
- If score ≤ 0.5 and groundtruth == 0 → Full credit
- Otherwise → No credit

In other words, 0.5 is the decision threshold.

- Each CV contributes equally.
- Final score = (number of correct decisions) / 5


In [ ]:
# =====================================================
#  Evaluation code
# =====================================================

def evaluate(scores, groundtruth, threshold=0.5):
    """
    scores: list of floats in [0, 1], length = 5
    groundtruth: list of ints (0 or 1), length = 5
    """
    assert len(scores) == 5
    assert len(groundtruth) == 5

    correct = 0
    decisions = []

    for s, gt in zip(scores, groundtruth):
        pred = 1 if s > threshold else 0
        decisions.append(pred)
        if pred == gt:
            correct += 1

    final_score = correct / len(scores)

    return {
        "decisions": decisions,
        "correct": correct,
        "total": len(scores),
        "final_score": final_score
    }


In [ ]:
# scores = ... # Your code should generate this list [0.2, 0.3, 0.4, 0.5, 0.6]
groundtruth = [1, 1, 1, 0, 0] # Do not modify

result = evaluate(all_scores, groundtruth)
print(result)